# 05 — Inference

Real, in-process inference (no HTTP call, works fully offline) using the exact same `ModelRegistry` + `predict_sentiment` path the FastAPI backend uses at request time (`backend/app/services/sentiment_service.py`, `backend/app/services/model_registry.py`).

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
from app.services.model_registry import ModelRegistry
from app.services.sentiment_service import predict_sentiment

registry = ModelRegistry()
registry.load_all()   # loads every artifact that's enabled in .env, once
print("Loaded:", {k: v.status for k, v in registry.statuses.items()})


## Single-review prediction

In [ ]:
examples = [
    "The product arrived early and works perfectly, I'm very happy with it.",
    "This item arrived broken and the seller never responded.",
]
for text in examples:
    result = predict_sentiment(registry, text, model_name="cnn2d")
    print(f"{text[:60]!r:65s} -> {result}")


## Command-line equivalent (also offline, no HTTP)

```bash
python backend/scripts/inference.py --text "The product arrived early and works perfectly." --model bert
```

Same `load_fine_tuned_bert` / `load_cnn2d_model` / `predict_sentiment` functions used above -- see `backend/scripts/inference.py`.

## Batch inference

`predict_sentiment_batch(registry, items, model_name=...)` -- same function the `/api/v1/sentiment/predict-batch` endpoint calls, preserving each input item's `id`.

In [ ]:
from app.services.sentiment_service import predict_sentiment_batch

items = [{"id": "r1", "text": examples[0]}, {"id": "r2", "text": examples[1]}]
for r in predict_sentiment_batch(registry, items, model_name="cnn2d"):
    print(r)
